<a href="https://colab.research.google.com/github/AllyApitchaya/msc-adr-prediction/blob/main/notebooks/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/MSc_ADR_Project'
print(f"Project path: {PROJECT_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project path: /content/drive/MyDrive/MSc_ADR_Project


In [ ]:
!pip install pubchempy -q
print("pubchempy installed")

pubchempy installed


In [ ]:
import pubchempy as pcp

drug_name = "metformin"
compounds = pcp.get_compounds(drug_name, 'name')

if compounds:
    c = compounds[0]
    print(f"Drug:              {drug_name}")
    print(f"PubChem CID:       {c.cid}")
    print(f"SMILES:            {c.connectivity_smiles}")
    print(f"Molecular formula: {c.molecular_formula}")
    print(f"Molecular weight:  {c.molecular_weight}")
else:
    print(f"Compound '{drug_name}' not found")

Drug:              metformin
PubChem CID:       4091
SMILES:            CN(C)C(=N)N=C(N)N
Molecular formula: C4H11N5
Molecular weight:  129.16


In [ ]:
import pubchempy as pcp
import pandas as pd
import time

# Antidiabetic drugs (ATC class A10), grouped by drug class
antidiabetic_drugs = [
    "metformin",       # Biguanide
    "glipizide",       # Sulfonylurea
    "glyburide",       # Sulfonylurea
    "glimepiride",     # Sulfonylurea
    "pioglitazone",    # Thiazolidinedione
    "rosiglitazone",   # Thiazolidinedione
    "sitagliptin",     # DPP-4 inhibitor
    "saxagliptin",     # DPP-4 inhibitor
    "linagliptin",     # DPP-4 inhibitor
    "empagliflozin",   # SGLT2 inhibitor
    "canagliflozin",   # SGLT2 inhibitor
    "dapagliflozin",   # SGLT2 inhibitor
    "repaglinide",     # Meglitinide
    "acarbose",        # Alpha-glucosidase inhibitor
    "nateglinide",     # Meglitinide
]

REQUEST_DELAY = 0.3  # seconds between requests to respect PubChem rate limits

records = []
failed = []

for drug in antidiabetic_drugs:
    try:
        compounds = pcp.get_compounds(drug, 'name')
        if compounds:
            c = compounds[0]
            records.append({
                'drug_name': drug,
                'pubchem_cid': c.cid,
                'smiles': c.connectivity_smiles,
                'formula': c.molecular_formula,
                'mol_weight': c.molecular_weight,
            })
            print(f"  retrieved: {drug} (CID {c.cid})")
        else:
            failed.append(drug)
            print(f"  not found: {drug}")
    except Exception as e:
        failed.append(drug)
        print(f"  error:     {drug} - {e}")
    time.sleep(REQUEST_DELAY)

df = pd.DataFrame(records)
print(f"\nRetrieved {len(df)} of {len(antidiabetic_drugs)} compounds")
if failed:
    print(f"Failed: {failed}")

df

  retrieved: metformin (CID 4091)
  retrieved: glipizide (CID 3478)
  retrieved: glyburide (CID 3488)
  retrieved: glimepiride (CID 3476)
  retrieved: pioglitazone (CID 4829)
  retrieved: rosiglitazone (CID 77999)
  retrieved: sitagliptin (CID 4369359)
  retrieved: saxagliptin (CID 11243969)
  retrieved: linagliptin (CID 10096344)
  retrieved: empagliflozin (CID 11949646)
  retrieved: canagliflozin (CID 24812758)
  retrieved: dapagliflozin (CID 9887712)
  retrieved: repaglinide (CID 65981)
  retrieved: acarbose (CID 9811704)
  retrieved: nateglinide (CID 5311309)

Retrieved 15 of 15 compounds


,drug_name,pubchem_cid,smiles,formula,mol_weight
0,metformin,4091,CN(C)C(=N)N=C(N)N,C4H11N5,129.16
1,glipizide,3478,CC1=CN=C(C=N1)C(=O)NCCC2=CC=C(C=C2)S(=O)(=O)NC...,C21H27N5O4S,445.50
2,glyburide,3488,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,C23H28ClN3O5S,494.00
3,glimepiride,3476,CCC1=C(CN(C1=O)C(=O)NCCC2=CC=C(C=C2)S(=O)(=O)N...,C24H34N4O5S,490.60
4,pioglitazone,4829,CCC1=CN=C(C=C1)CCOC2=CC=C(C=C2)CC3C(=O)NC(=O)S3,C19H20N2O3S,356.40
5,rosiglitazone,77999,CN(CCOC1=CC=C(C=C1)CC2C(=O)NC(=O)S2)C3=CC=CC=N3,C18H19N3O3S,357.40
6,sitagliptin,4369359,C1CN2C(=NN=C2C(F)(F)F)CN1C(=O)CC(CC3=CC(=C(C=C...,C16H15F6N5O,407.31
7,saxagliptin,11243969,C1C2CC2N(C1C#N)C(=O)C(C34CC5CC(C3)CC(C5)(C4)O)N,C18H25N3O2,315.40
8,linagliptin,10096344,CC#CCN1C2=C(N=C1N3CCCC(C3)N)N(C(=O)N(C2=O)CC4=...,C25H28N8O2,472.50
9,empagliflozin,11949646,C1COCC1OC2=CC=C(C=C2)CC3=C(C=CC(=C3)C4C(C(C(C(...,C23H27ClO7,450.90


In [ ]:
import os

output_dir = os.path.join(PROJECT_PATH, 'data', 'raw')
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'antidiabetic_drugs_smiles.csv')
df.to_csv(output_path, index=False)

print(f"Saved {len(df)} records to: {output_path}")
print()
print(df[['drug_name', 'pubchem_cid', 'formula']].to_string(index=False))

Saved 15 records to: /content/drive/MyDrive/MSc_ADR_Project/data/raw/antidiabetic_drugs_smiles.csv

    drug_name  pubchem_cid       formula
    metformin         4091       C4H11N5
    glipizide         3478   C21H27N5O4S
    glyburide         3488 C23H28ClN3O5S
  glimepiride         3476   C24H34N4O5S
 pioglitazone         4829   C19H20N2O3S
rosiglitazone        77999   C18H19N3O3S
  sitagliptin      4369359   C16H15F6N5O
  saxagliptin     11243969    C18H25N3O2
  linagliptin     10096344    C25H28N8O2
empagliflozin     11949646    C23H27ClO7
canagliflozin     24812758    C24H25FO5S
dapagliflozin      9887712    C21H25ClO6
  repaglinide        65981    C27H36N2O4
     acarbose      9811704    C25H43NO18
  nateglinide      5311309     C19H27NO3


In [ ]:
!pip install rdkit -q
print("rdkit installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 48.8 MB/s eta 0:00:00
rdkit installed


In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

# RDKit is pre-installed in Colab; verify import
print(f"RDKit version: {Chem.rdBase.rdkitVersion}\n")

valid_count = 0
invalid = []

for _, row in df.iterrows():
    mol = Chem.MolFromSmiles(row['smiles'])
    if mol is not None:
        n_atoms = mol.GetNumAtoms()
        n_bonds = mol.GetNumBonds()
        mw = Descriptors.MolWt(mol)
        print(f"  {row['drug_name']:16s} atoms={n_atoms:3d}  bonds={n_bonds:3d}  MW={mw:7.2f}")
        valid_count += 1
    else:
        invalid.append(row['drug_name'])
        print(f"  {row['drug_name']:16s} FAILED to parse")

print(f"\n{valid_count} of {len(df)} SMILES parsed successfully")
if invalid:
    print(f"Invalid: {invalid}")
else:
    print("All SMILES are valid and ready for graph construction")

RDKit version: 2026.03.2

  metformin        atoms=  9  bonds=  8  MW= 129.17
  glipizide        atoms= 31  bonds= 33  MW= 445.55
  glyburide        atoms= 33  bonds= 35  MW= 494.01
  glimepiride      atoms= 34  bonds= 36  MW= 490.63
  pioglitazone     atoms= 25  bonds= 27  MW= 356.45
  rosiglitazone    atoms= 25  bonds= 27  MW= 357.44
  sitagliptin      atoms= 28  bonds= 30  MW= 407.32
  saxagliptin      atoms= 23  bonds= 27  MW= 315.42
  linagliptin      atoms= 35  bonds= 39  MW= 472.55
  empagliflozin    atoms= 31  bonds= 34  MW= 450.92
  canagliflozin    atoms= 31  bonds= 34  MW= 444.52
  dapagliflozin    atoms= 28  bonds= 30  MW= 408.88
  repaglinide      atoms= 33  bonds= 35  MW= 452.60
  acarbose         atoms= 44  bonds= 46  MW= 645.61
  nateglinide      atoms= 23  bonds= 24  MW= 317.43

15 of 15 SMILES parsed successfully
All SMILES are valid and ready for graph construction


In [ ]:
import os
import urllib.request

# FAERS quarterly data follows the URL pattern:
#   https://fis.fda.gov/content/Exports/faers_ascii_YYYYqQ.zip
# Start with a single quarter (2020 Q1) to inspect the data structure
# before downloading the full study period.

FAERS_DIR = os.path.join(PROJECT_PATH, 'data', 'raw', 'faers')
os.makedirs(FAERS_DIR, exist_ok=True)

quarter = '2020q1'
url = f'https://fis.fda.gov/content/Exports/faers_ascii_{quarter}.zip'
zip_path = os.path.join(FAERS_DIR, f'faers_ascii_{quarter}.zip')

if os.path.exists(zip_path):
    print(f"Already downloaded: {zip_path}")
else:
    print(f"Downloading {url}")
    urllib.request.urlretrieve(url, zip_path)
    print("Download complete")

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"File size: {size_mb:.1f} MB")

Download complete
File size: 65.2 MB


In [ ]:
import zipfile

extract_dir = os.path.join(FAERS_DIR, quarter)
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

print(f"Extracted to: {extract_dir}\n")
print("Contents:")
for root, dirs, files in os.walk(extract_dir):
    for f in sorted(files):
        full = os.path.join(root, f)
        size_mb = os.path.getsize(full) / (1024 * 1024)
        rel = os.path.relpath(full, extract_dir)
        print(f"  {rel:40s} {size_mb:7.2f} MB")

Extracted to: /content/drive/MyDrive/MSc_ADR_Project/data/raw/faers/2020q1

Contents:
  FAQs.pdf                                    0.17 MB
  Readme.pdf                                  0.10 MB
  ASCII/ASC_NTS.pdf                           0.29 MB
  ASCII/DEMO20Q1.txt                         61.95 MB
  ASCII/DRUG20Q1.txt                        177.10 MB
  ASCII/INDI20Q1.txt                         60.58 MB
  ASCII/OUTC20Q1.txt                          7.04 MB
  ASCII/REAC20Q1.txt                         53.82 MB
  ASCII/RPSR20Q1.txt                          0.33 MB
  ASCII/THER20Q1.PDF                          0.03 MB
  ASCII/THER20Q1.txt                         24.66 MB
  ASCII/demo20q1.pdf                          0.09 MB
  ASCII/drug20q1.pdf                          0.15 MB
  ASCII/indi20q1.pdf                          0.04 MB
  ASCII/outc20q1.pdf                          0.05 MB
  ASCII/reac20q1.pdf                          0.04 MB
  ASCII/rpsr20q1.pdf                          0.04

In [ ]:
import pandas as pd
import glob

# Locate the ASCII directory (FDA stores the 7 data files inside it)
ascii_dir = None
for root, dirs, files in os.walk(extract_dir):
    if any(f.upper().startswith('DEMO') for f in files):
        ascii_dir = root
        break

print(f"ASCII directory: {ascii_dir}\n")

def load_faers_table(prefix):
    """Load one FAERS ASCII table. Files are dollar-sign delimited."""
    matches = glob.glob(os.path.join(ascii_dir, f'{prefix}*.txt'))
    if not matches:
        raise FileNotFoundError(f"No file starting with '{prefix}' found")
    return pd.read_csv(matches[0], sep='$', dtype=str, encoding='latin-1')

# DRUG: drugs reported in each case
# REAC: adverse reactions reported in each case
drug = load_faers_table('DRUG')
reac = load_faers_table('REAC')

print(f"DRUG table: {drug.shape[0]:,} rows, {drug.shape[1]} columns")
print(f"  columns: {list(drug.columns)}\n")
print(f"REAC table: {reac.shape[0]:,} rows, {reac.shape[1]} columns")
print(f"  columns: {list(reac.columns)}")

ASCII directory: /content/drive/MyDrive/MSc_ADR_Project/data/raw/faers/2020q1/ASCII

DRUG table: 1,943,532 rows, 20 columns
  columns: ['primaryid', 'caseid', 'drug_seq', 'role_cod', 'drugname', 'prod_ai', 'val_vbm', 'route', 'dose_vbm', 'cum_dose_chr', 'cum_dose_unit', 'dechal', 'rechal', 'lot_num', 'exp_dt', 'nda_num', 'dose_amt', 'dose_unit', 'dose_form', 'dose_freq']

REAC table: 1,517,264 rows, 4 columns
  columns: ['primaryid', 'caseid', 'pt', 'drug_rec_act']


In [ ]:
print("=== DRUG table (first 5 rows) ===")
print(drug.head().to_string())
print()
print("=== REAC table (first 5 rows) ===")
print(reac.head().to_string())

=== DRUG table (first 5 rows) ===
   primaryid    caseid drug_seq role_cod               drugname               prod_ai val_vbm route          dose_vbm cum_dose_chr cum_dose_unit dechal rechal lot_num exp_dt nda_num dose_amt dose_unit           dose_form dose_freq
0  100046942  10004694        1       PS                LIPITOR  ATORVASTATIN CALCIUM       1  Oral      20 MG, DAILY          NaN           NaN      U    NaN     NaN    NaN  020702       20        MG  FILM-COATED TABLET       NaN
1  100046942  10004694        2       SS                LIPITOR  ATORVASTATIN CALCIUM       1   NaN               NaN          NaN           NaN      U    NaN     NaN    NaN  020702      NaN       NaN  FILM-COATED TABLET       NaN
2  100046942  10004694        3       SS  ATORVASTATIN CALCIUM.  ATORVASTATIN CALCIUM       1  Oral      20 MG, DAILY          NaN           NaN      U    NaN     NaN    NaN  020702       20        MG  FILM-COATED TABLET       NaN
3  100046942  10004694        4        C  

In [ ]:
# Load the antidiabetic drug list collected earlier from PubChem
drug_list_path = os.path.join(PROJECT_PATH, 'data', 'raw',
                               'antidiabetic_drugs_smiles.csv')
drugs_df = pd.read_csv(drug_list_path)
target_drugs = [d.lower() for d in drugs_df['drug_name'].tolist()]

print(f"Target antidiabetic drugs ({len(target_drugs)}):")
print(f"  {target_drugs}\n")

# FAERS records drug names in free text and may use brand names,
# so match against the 'drugname' field using a case-insensitive
# substring search for each target generic name.
drug_lower = drug['drugname'].str.lower().fillna('')

mask = drug_lower.apply(
    lambda name: any(target in name for target in target_drugs)
)
drug_diabetes = drug[mask].copy()

print(f"DRUG rows matching antidiabetic drugs: {len(drug_diabetes):,} "
      f"(of {len(drug):,} total)")
print(f"\nMatched drug-name variants:")
print(drug_diabetes['drugname'].value_counts().head(20).to_string())

Target antidiabetic drugs (15):
  ['metformin', 'glipizide', 'glyburide', 'glimepiride', 'pioglitazone', 'rosiglitazone', 'sitagliptin', 'saxagliptin', 'linagliptin', 'empagliflozin', 'canagliflozin', 'dapagliflozin', 'repaglinide', 'acarbose', 'nateglinide']

DRUG rows matching antidiabetic drugs: 17,196 (of 1,943,532 total)

Matched drug-name variants:
drugname
METFORMIN                   9972
GLIPIZIDE.                  1242
GLIMEPIRIDE.                1082
METFORMIN HYDROCHLORIDE.     711
METFORMIN HCL                542
GLYBURIDE.                   303
PIOGLITAZONE.                264
METFORMINE                   225
SITAGLIPTIN                  225
DAPAGLIFLOZIN.               203
REPAGLINIDE.                 196
EMPAGLIFLOZIN                161
LINAGLIPTIN                  135
CANAGLIFLOZIN                124
ACARBOSE.                    107
Metformin tablet              93
METFORMIN ER                  77
METFORMINE [METFORMIN]        67
Sitagliptin                   64
SITAGLI

In [ ]:
# Keep only Primary Suspect (PS) records: the drug most likely
# responsible for the adverse event. This reduces noise from
# concomitant medications, following standard FAERS methodology.
drug_ps = drug_diabetes[drug_diabetes['role_cod'] == 'PS'].copy()
print(f"Primary-suspect antidiabetic records: {len(drug_ps):,}")

# Join with the reaction table on primaryid to pair each
# drug record with its reported adverse reactions.
merged = drug_ps[['primaryid', 'caseid', 'drugname', 'role_cod']].merge(
    reac[['primaryid', 'pt']],
    on='primaryid',
    how='inner',
)

print(f"Drug-ADR pairs after join: {len(merged):,}")
print(f"Unique cases: {merged['primaryid'].nunique():,}")
print(f"Unique ADR terms: {merged['pt'].nunique():,}\n")

print("Most frequent ADRs for antidiabetic drugs:")
print(merged['pt'].value_counts().head(15).to_string())

Primary-suspect antidiabetic records: 2,708
Drug-ADR pairs after join: 9,754
Unique cases: 2,708
Unique ADR terms: 1,301

Most frequent ADRs for antidiabetic drugs:
pt
Lactic acidosis                      417
Acute kidney injury                  290
Diarrhoea                            237
Drug ineffective                     154
Hypoglycaemia                        153
Metabolic acidosis                   149
Nausea                               135
Vomiting                             128
Toxicity to various agents           102
Completed suicide                     97
Malaise                               93
Euglycaemic diabetic ketoacidosis     92
Fatigue                               87
Dyspnoea                              79
Hypotension                           78


In [ ]:
# Save the processed drug-ADR pairs for this quarter.
# These are stored in data/processed since they are derived
# (filtered and joined), not raw downloads.
processed_dir = os.path.join(PROJECT_PATH, 'data', 'processed')
os.makedirs(processed_dir, exist_ok=True)

output_path = os.path.join(processed_dir, f'faers_{quarter}_diabetes_adr.csv')
merged.to_csv(output_path, index=False)

print(f"Saved {len(merged):,} drug-ADR pairs to:")
print(f"  {output_path}")
print()
print("Summary for", quarter)
print(f"  Primary-suspect records: {len(drug_ps):,}")
print(f"  Drug-ADR pairs:          {len(merged):,}")
print(f"  Unique cases:            {merged['primaryid'].nunique():,}")
print(f"  Unique ADR terms:        {merged['pt'].nunique():,}")

Saved 9,754 drug-ADR pairs to:
  /content/drive/MyDrive/MSc_ADR_Project/data/processed/faers_2020q1_diabetes_adr.csv

Summary for 2020q1
  Primary-suspect records: 2,708
  Drug-ADR pairs:          9,754
  Unique cases:            2,708
  Unique ADR terms:        1,301


In [ ]:
import os
import glob
import zipfile
import urllib.request
import pandas as pd

# Reusable pipeline for a single FAERS quarter:
# download, extract, filter to antidiabetic primary-suspect
# records, and join with the reaction table.

FAERS_BASE_URL = 'https://fis.fda.gov/content/Exports'

# Antidiabetic generic names (loaded once, reused for every quarter)
_drug_list_path = os.path.join(PROJECT_PATH, 'data', 'raw',
                               'antidiabetic_drugs_smiles.csv')
TARGET_DRUGS = [d.lower() for d in
                pd.read_csv(_drug_list_path)['drug_name'].tolist()]


def _load_faers_table(ascii_dir, prefix):
    """Load one dollar-sign delimited FAERS ASCII table."""
    matches = glob.glob(os.path.join(ascii_dir, f'{prefix}*.txt'))
    if not matches:
        raise FileNotFoundError(f"No '{prefix}' file in {ascii_dir}")
    return pd.read_csv(matches[0], sep='$', dtype=str, encoding='latin-1')


def process_quarter(quarter, faers_dir, keep_zip=False):
    """Download and process one FAERS quarter.

    Returns a DataFrame of antidiabetic drug-ADR pairs for the
    quarter, or None if the download fails.
    """
    url = f'{FAERS_BASE_URL}/faers_ascii_{quarter}.zip'
    zip_path = os.path.join(faers_dir, f'faers_ascii_{quarter}.zip')
    extract_dir = os.path.join(faers_dir, quarter)

    # 1. Download (skip if already present)
    if not os.path.exists(zip_path):
        try:
            urllib.request.urlretrieve(url, zip_path)
        except Exception as e:
            print(f"  {quarter}: download failed - {e}")
            return None

    # 2. Extract
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)

    # 3. Locate the ASCII directory
    ascii_dir = None
    for root, dirs, files in os.walk(extract_dir):
        if any(f.upper().startswith('DEMO') for f in files):
            ascii_dir = root
            break

    # 4. Load DRUG and REAC tables
    drug = _load_faers_table(ascii_dir, 'DRUG')
    reac = _load_faers_table(ascii_dir, 'REAC')

    # 5. Filter to antidiabetic primary-suspect records
    name_lower = drug['drugname'].str.lower().fillna('')
    is_target = name_lower.apply(
        lambda n: any(t in n for t in TARGET_DRUGS))
    drug_ps = drug[is_target & (drug['role_cod'] == 'PS')].copy()

    # 6. Join with reactions
    merged = drug_ps[['primaryid', 'caseid', 'drugname', 'role_cod']].merge(
        reac[['primaryid', 'pt']], on='primaryid', how='inner')
    merged['quarter'] = quarter

    # 7. Clean up large files to save disk space
    if not keep_zip and os.path.exists(zip_path):
        os.remove(zip_path)

    return merged


print("process_quarter() defined")
print(f"Target drugs loaded: {len(TARGET_DRUGS)}")

process_quarter() defined
Target drugs loaded: 15


In [ ]:
import time

# Study period from the project proposal: 2020-2022
QUARTERS = [f'{year}q{q}'
            for year in (2020, 2021, 2022)
            for q in (1, 2, 3, 4)]

print(f"Processing {len(QUARTERS)} quarters: {QUARTERS}\n")

all_quarters = []
for i, q in enumerate(QUARTERS, 1):
    start = time.time()
    result = process_quarter(q, FAERS_DIR, keep_zip=False)
    if result is not None:
        all_quarters.append(result)
        elapsed = time.time() - start
        print(f"  [{i:2d}/{len(QUARTERS)}] {q}: "
              f"{len(result):,} drug-ADR pairs  ({elapsed:.0f}s)")
    else:
        print(f"  [{i:2d}/{len(QUARTERS)}] {q}: SKIPPED")

# Combine all quarters into a single DataFrame
faers_all = pd.concat(all_quarters, ignore_index=True)
print(f"\nCombined: {len(faers_all):,} drug-ADR pairs "
      f"across {len(all_quarters)} quarters")

Processing 12 quarters: ['2020q1', '2020q2', '2020q3', '2020q4', '2021q1', '2021q2', '2021q3', '2021q4', '2022q1', '2022q2', '2022q3', '2022q4']

  [ 1/12] 2020q1: 9,754 drug-ADR pairs  (21s)
  [ 2/12] 2020q2: 7,141 drug-ADR pairs  (141s)
  [ 3/12] 2020q3: 7,906 drug-ADR pairs  (144s)
  [ 4/12] 2020q4: 7,489 drug-ADR pairs  (156s)
  [ 5/12] 2021q1: 8,060 drug-ADR pairs  (151s)
  [ 6/12] 2021q2: 6,028 drug-ADR pairs  (150s)
  [ 7/12] 2021q3: 6,391 drug-ADR pairs  (160s)
  [ 8/12] 2021q4: 7,641 drug-ADR pairs  (132s)
  [ 9/12] 2022q1: 7,717 drug-ADR pairs  (143s)
  [10/12] 2022q2: 7,584 drug-ADR pairs  (141s)
  [11/12] 2022q3: 6,913 drug-ADR pairs  (134s)
  [12/12] 2022q4: 8,199 drug-ADR pairs  (152s)

Combined: 90,823 drug-ADR pairs across 12 quarters


In [ ]:
print("Before deduplication:")
print(f"  Total drug-ADR pairs: {len(faers_all):,}")
print(f"  Unique cases (caseid): {faers_all['caseid'].nunique():,}")
print(f"  Unique primaryids:     {faers_all['primaryid'].nunique():,}")

# FAERS deduplication
# A case (caseid) may be reported multiple times across quarters as
# follow-up reports. Each submission gets a new primaryid but keeps
# the same caseid. We keep only the latest primaryid per caseid,
# since the primaryid increases with each follow-up.
faers_all['primaryid_num'] = pd.to_numeric(faers_all['primaryid'],
                                           errors='coerce')

latest = (faers_all
          .sort_values('primaryid_num')
          .drop_duplicates(subset='caseid', keep='last'))

# After selecting the latest report per case, we still re-join its
# drug-ADR pairs and drop any exact duplicate pairs.
latest_ids = set(latest['primaryid'])
faers_dedup = faers_all[faers_all['primaryid'].isin(latest_ids)].copy()
faers_dedup = faers_dedup.drop_duplicates(
    subset=['caseid', 'drugname', 'pt'])
faers_dedup = faers_dedup.drop(columns='primaryid_num')

print("\nAfter deduplication:")
print(f"  Total drug-ADR pairs: {len(faers_dedup):,}")
print(f"  Unique cases:         {faers_dedup['caseid'].nunique():,}")
print(f"  Unique ADR terms:     {faers_dedup['pt'].nunique():,}")
print(f"  Removed: {len(faers_all) - len(faers_dedup):,} duplicate pairs")

Before deduplication:
  Total drug-ADR pairs: 90,823
  Unique cases (caseid): 23,218
  Unique primaryids:     24,719

After deduplication:
  Total drug-ADR pairs: 80,958
  Unique cases:         23,218
  Unique ADR terms:     3,586
  Removed: 9,865 duplicate pairs


In [ ]:
output_path = os.path.join(PROJECT_PATH, 'data', 'processed',
                           'faers_2020_2022_diabetes_adr.csv')
faers_dedup.to_csv(output_path, index=False)

print(f"Saved {len(faers_dedup):,} drug-ADR pairs to:")
print(f"  {output_path}\n")

# Distribution across quarters
print("Pairs per quarter:")
print(faers_dedup['quarter'].value_counts().sort_index().to_string())

print("\nTop 15 ADRs across the full study period:")
print(faers_dedup['pt'].value_counts().head(15).to_string())

Saved 80,958 drug-ADR pairs to:
  /content/drive/MyDrive/MSc_ADR_Project/data/processed/faers_2020_2022_diabetes_adr.csv

Pairs per quarter:
quarter
2020q1    8210
2020q2    6127
2020q3    6975
2020q4    6444
2021q1    6934
2021q2    5330
2021q3    5712
2021q4    6960
2022q1    6689
2022q2    6855
2022q3    6592
2022q4    8130

Top 15 ADRs across the full study period:
pt
Lactic acidosis                      4312
Acute kidney injury                  2655
Metabolic acidosis                   1703
Diarrhoea                            1646
Toxicity to various agents           1493
Hypoglycaemia                        1208
Drug ineffective                     1010
Nausea                               1006
Vomiting                              983
Blood glucose increased               679
Fatigue                               663
Hypotension                           660
Diabetic ketoacidosis                 655
Overdose                              640
Euglycaemic diabetic ketoacidosis    

In [ ]:
import pandas as pd
import os

# Load the deduplicated FAERS data and the PubChem drug list
faers_path = os.path.join(PROJECT_PATH, 'data', 'processed',
                          'faers_2020_2022_diabetes_adr.csv')
faers = pd.read_csv(faers_path)

pubchem_path = os.path.join(PROJECT_PATH, 'data', 'raw',
                            'antidiabetic_drugs_smiles.csv')
pubchem = pd.read_csv(pubchem_path)

# FAERS records drug names inconsistently (brand names, salts,
# misspellings). Map each FAERS record to a standard generic name
# by checking which target generic name appears in the free text.
generics = pubchem['drug_name'].str.lower().tolist()

def map_to_generic(faers_name):
    name = str(faers_name).lower()
    for generic in generics:
        if generic in name:
            return generic
    return None

faers['generic_name'] = faers['drugname'].apply(map_to_generic)

unmapped = faers['generic_name'].isna().sum()
print(f"Mapped:   {(faers['generic_name'].notna()).sum():,} rows")
print(f"Unmapped: {unmapped:,} rows")
print()
print("Records per generic drug:")
print(faers['generic_name'].value_counts().to_string())

Mapped:   80,958 rows
Unmapped: 0 rows

Records per generic drug:
generic_name
metformin        67432
dapagliflozin     4999
empagliflozin     2343
glimepiride       1074
pioglitazone      1020
glipizide          900
linagliptin        859
canagliflozin      776
sitagliptin        620
repaglinide        442
glyburide          193
saxagliptin        174
acarbose           107
nateglinide         12
rosiglitazone        7


In [ ]:
# Drop any rows that could not be mapped to a target generic name
faers_mapped = faers.dropna(subset=['generic_name']).copy()

# Join with PubChem to attach molecular structure (SMILES) to
# every drug-ADR pair.
final = faers_mapped.merge(
    pubchem[['drug_name', 'smiles', 'pubchem_cid', 'formula']],
    left_on='generic_name',
    right_on='drug_name',
    how='inner',
)

# Keep the columns relevant for modelling
final = final[['generic_name', 'pubchem_cid', 'smiles',
               'pt', 'caseid', 'quarter']].rename(
    columns={'pt': 'adr', 'generic_name': 'drug'})

print(f"Final dataset: {len(final):,} drug-ADR pairs")
print(f"  Unique drugs: {final['drug'].nunique()}")
print(f"  Unique ADRs:  {final['adr'].nunique():,}")
print(f"  Unique cases: {final['caseid'].nunique():,}\n")

# Save
output_path = os.path.join(PROJECT_PATH, 'data', 'processed',
                           'drug_adr_with_structures.csv')
final.to_csv(output_path, index=False)
print(f"Saved to: {output_path}\n")

print("Sample rows:")
print(final.head(10).to_string(index=False))

Final dataset: 80,958 drug-ADR pairs
  Unique drugs: 15
  Unique ADRs:  3,586
  Unique cases: 23,218

Saved to: /content/drive/MyDrive/MSc_ADR_Project/data/processed/drug_adr_with_structures.csv

Sample rows:
     drug  pubchem_cid            smiles                                              adr   caseid quarter
metformin         4091 CN(C)C(=N)N=C(N)N                                 Cardiac disorder 10082646  2020q1
metformin         4091 CN(C)C(=N)N=C(N)N                                 Drug interaction 10082646  2020q1
metformin         4091 CN(C)C(=N)N=C(N)N             Diabetes mellitus inadequate control 10265285  2020q1
metformin         4091 CN(C)C(=N)N=C(N)N                                 Drug ineffective 10265285  2020q1
metformin         4091 CN(C)C(=N)N=C(N)N                                    Hypoglycaemia 10265285  2020q1
metformin         4091 CN(C)C(=N)N=C(N)N Inappropriate schedule of product administration 10265285  2020q1
metformin         4091 CN(C)C(=N)N=C(N)N  